# ➕ Python Prefix Sum — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Prefix sum is like a running odometer on a road trip.
> Each city records its total distance from the start.
> To find the distance between city A and city B, just subtract their odometer readings.
> You never re-drive the road — one pass to build, O(1) to query.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is Prefix Sum? The Visual Model](#1) |
| 2 | [Building a Prefix Array](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Range Sum Query — LC 303](#5) |
| 6 | [Pattern 2: Subarray Sum Equals K — LC 560](#6) |
| 7 | [Pattern 3: Product Except Self — LC 238](#7) |
| 8 | [Pattern 4: 2D Prefix Sum — LC 304](#8) |
| 9 | [Pattern 5: Prefix XOR — count subarrays with even XOR](#9) |
| 10 | [The Prefix Sum Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Is Prefix Sum? The Visual Model

```
BUILDING THE PREFIX ARRAY

  nums = [3,  0,  1,  4,  2,  5]
  idx:  [ 0   1   2   3   4   5]

  pre  = [0,  3,  3,  4,  8, 10, 15]
  idx:  [ 0   1   2   3   4   5   6]

  pre[i] = sum(nums[0..i-1])
  pre[0] = 0  (empty prefix, the seed)

RANGE QUERY: sum(nums[i..j]) = pre[j+1] - pre[i]

  sum(nums[2..4]) = pre[5] - pre[2] = 10 - 3 = 7
  Verify: nums[2]+nums[3]+nums[4] = 1+4+2 = 7 ✓

2D PREFIX SUM — tile diagram

  grid:         prefix at (2,2):
  [1  2  3]     pre[r][c] = grid[r][c]
  [4  5  6]               + pre[r-1][c]
  [7  8  9]               + pre[r][c-1]
                          - pre[r-1][c-1]  (remove double-counted corner)

HASHMAP + PREFIX SUM (LC 560):
  Want subarrays where sum == k.
  pre[j] - pre[i] = k  →  pre[i] = pre[j] - k
  Seed map with {0: 1} to count subarrays starting at index 0.

WHY O(1) QUERY:
  One pass builds pre[]. Each query is two array lookups and a subtraction.
```

<a id='2'></a>

## 2. Building a Prefix Array

In [ ]:
from itertools import accumulate

nums = [3, 0, 1, 4, 2, 5]

# METHOD 1: manual loop (most explicit, easiest to understand)
pre = [0] * (len(nums) + 1)   # extra slot at front for the empty-prefix seed
for i, v in enumerate(nums):
    pre[i + 1] = pre[i] + v   # each entry is previous + current
print(f"manual prefix:      {pre}")

# METHOD 2: itertools.accumulate
pre2 = [0] + list(accumulate(nums))
print(f"accumulate prefix:  {pre2}")

# METHOD 3: prefix XOR (for XOR problems)
pre_xor = [0] * (len(nums) + 1)
for i, v in enumerate(nums):
    pre_xor[i + 1] = pre_xor[i] ^ v
print(f"prefix XOR:         {pre_xor}")

# QUERY: sum of nums[2..4]
i, j = 2, 4
query_result = pre[j + 1] - pre[i]
print(f"\nsum(nums[{i}..{j}]) = pre[{j+1}] - pre[{i}] = {pre[j+1]} - {pre[i]} = {query_result}")
print(f"verify: {sum(nums[i:j+1])}")
print("Prefix array forms demonstrated.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                          COMPLEXITY   WHAT IT DOES
────────────────────────────────────────────────────────────────────
build prefix array                 O(n)         one pass to build pre[]
range sum query sum(i..j)          O(1)         pre[j+1] - pre[i]
range XOR query xor(i..j)          O(1)         pre_xor[j+1] ^ pre_xor[i]
build 2D prefix                    O(m*n)       one pass row by row
2D range query                     O(1)         inclusion-exclusion formula
prefix + HashMap for subarray k    O(n)         count prefix occurrences

THINGS YOU DO NOT DO:
❌  Forget the seed: pre[0] = 0 (handles subarrays starting at index 0)
❌  Use 0-indexed prefix without the +1 offset (off-by-one errors)
❌  Use prefix sum on non-integer data without defining the aggregation
❌  Query sum(i..j) as pre[j] - pre[i] — wrong by one element
❌  Build 2D prefix without the inclusion-exclusion step
```

In [ ]:
# Live demo: build and query prefix sum
nums = [3, 0, 1, 4, 2, 5]
pre = [0] + list(__import__('itertools').accumulate(nums))

print("Range queries (all O(1) after O(n) build):")
for i, j in [(0, 5), (1, 3), (2, 4), (0, 0)]:
    result = pre[j + 1] - pre[i]
    print(f"  sum({i}..{j}) = pre[{j+1}]({pre[j+1]}) - pre[{i}]({pre[i]}) = {result}")

print()

# Demo: HashMap pattern for 'prefix seen before'
from collections import defaultdict

nums2 = [1, 1, 1]
k = 2
count_map = defaultdict(int)
count_map[0] = 1      # seed: empty prefix = sum 0, seen once
running = 0
total = 0

print(f"Finding subarrays summing to {k} in {nums2}:")
for i, v in enumerate(nums2):
    running += v
    complement = running - k   # how many times have we seen (running-k)?
    total += count_map[complement]
    count_map[running] += 1
    print(f"  i={i} v={v} running={running} need={complement} found={count_map[complement]} total={total}")

print(f"Answer: {total} subarrays sum to {k}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    WHAT TO DO
────────────────────────────────────────────────────────────────────
"sum of subarray / range query"          Build prefix array, O(1) query
"count subarrays summing to k"           Prefix + HashMap, seed {0:1}
"product of all except self"             Left-pass prefix × right-pass suffix
"2D range sum query"                     2D prefix with inclusion-exclusion
"count subarrays with even XOR"          Prefix XOR + parity count
"subarray with sum divisible by k"       Prefix mod k + HashMap
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Range Sum Query — LC 303

---

```
PROBLEM:
  Build a structure that answers sumRange(i, j) in O(1) after O(n) build.

TRICK:
  Build pre[0..n] where pre[i] = sum(nums[0..i-1]).
  Answer: pre[j+1] - pre[i].

SLOW MOTION TRACE on nums=[-2,0,3,-5,2,-1]:
  pre = [0, -2, -2, 1, -4, -2, -3]

  sumRange(0,2) = pre[3]-pre[0] = 1-0 = 1
    verify: -2+0+3 = 1 ✓
  sumRange(2,5) = pre[6]-pre[2] = -3-(-2) = -1
    verify: 3+(-5)+2+(-1) = -1 ✓
  sumRange(0,5) = pre[6]-pre[0] = -3-0 = -3
    verify: -2+0+3-5+2-1 = -3 ✓

KEY INSIGHT:
  Pre-computing cumulative sums trades O(n) setup for O(1) per query.
  Essential when many range queries on static array.

TIME:  O(n) build, O(1) query
SPACE: O(n) — the prefix array
```

In [ ]:
from typing import List

class NumArray:
    """
    LC 303 — Range Sum Query - Immutable
    Approach: prefix sum array for O(1) range queries.
    Time:  O(n) init, O(1) sumRange
    Space: O(n) — prefix array of length n+1
    """
    def __init__(self, nums: List[int]):
        self.pre = [0] * (len(nums) + 1)   # seed: pre[0]=0 handles left boundary
        for i, v in enumerate(nums):
            self.pre[i + 1] = self.pre[i] + v

    def sumRange(self, left: int, right: int) -> int:
        return self.pre[right + 1] - self.pre[left]  # O(1) lookup

# Slow motion on [-2,0,3,-5,2,-1]:
# pre = [0, -2, -2, 1, -4, -2, -3]
# sumRange(0,2) = pre[3]-pre[0] = 1-0 = 1
# sumRange(2,5) = pre[6]-pre[2] = -3-(-2) = -1

def test_harness(cls):
    tests = [
        ([-2,0,3,-5,2,-1], [(0,2,1), (2,5,-1), (0,5,-3)]),
        ([1,2,3,4,5],      [(0,4,15),(1,3,9),(2,2,3)]),
        ([5],              [(0,0,5)]),
        ([-1,-2,-3],       [(0,2,-6),(0,1,-3)]),
    ]
    passed = total = 0
    for nums, queries in tests:
        obj = cls(nums)
        for l, r, expected in queries:
            got = obj.sumRange(l, r)
            total += 1
            if got == expected:
                passed += 1
            else:
                print(f"FAILED | nums={nums} query=({l},{r}) expected={expected} got={got}")
    print(f"{passed}/{total} tests passed")

test_harness(NumArray)
print("NumArray defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Subarray Sum Equals K — LC 560

---

```
PROBLEM:
  Count subarrays whose elements sum to k.

TRICK:
  pre[j] - pre[i] = k  →  pre[i] = pre[j] - k
  Keep a HashMap of prefix sums seen so far.
  At each j, count how many past i give the required pre[i].
  Seed {0: 1} for subarrays starting at index 0 (pre[0]=0 was seen once).

SLOW MOTION TRACE on nums=[1,1,1], k=2:
  map={0:1}, running=0, total=0

  i=0: v=1, running=1, need=1-2=-1, map[-1]=0, total+=0=0. map={0:1,1:1}
  i=1: v=1, running=2, need=2-2=0,  map[0]=1,  total+=1=1. map={0:1,1:2,2:1}
  i=2: v=1, running=3, need=3-2=1,  map[1]=2,  total+=2=3. map={...3:1}

  answer = 3   (subarrays [0,1],[1,2],[0,2])

KEY INSIGHT:
  The seed {0:1} is critical — it counts subarrays starting at index 0
  where running_sum == k directly.

TIME:  O(n) — one pass
SPACE: O(n) — prefix sum HashMap
```

In [ ]:
from collections import defaultdict

def subarray_sum(nums: List[int], k: int) -> int:
    """
    LC 560 — Subarray Sum Equals K
    Approach: prefix sum + HashMap. Count (pre[j] - k) occurrences.
    Args:
        nums (List[int]): integer array (may contain negatives).
        k (int): target subarray sum.
    Returns:
        int: count of subarrays summing to k.
    Time:  O(n) — one pass through nums
    Space: O(n) — HashMap stores at most n distinct prefix sums
    """
    prefix_count = defaultdict(int)
    prefix_count[0] = 1   # seed: empty prefix seen once (handles start-of-array)
    running = 0
    total = 0

    for v in nums:
        running += v
        complement = running - k      # if complement was a past prefix, we found k
        total += prefix_count[complement]
        prefix_count[running] += 1    # record this prefix for future use

    return total

# Slow motion on [1,1,1], k=2:
# map={0:1}, running=0
# v=1: running=1, need=-1→0, total=0, map={0:1,1:1}
# v=1: running=2, need=0→1,  total=1, map={0:1,1:1,2:1}
# v=1: running=3, need=1→1,  total=3, map={...}
# answer=3

def test_harness(fn):
    tests = [
        ([1,1,1],       2, 3),
        ([1,2,3],       3, 2),
        ([0,0,0,0,0],   0, 15),
        ([-1,-1,1],     0, 1),
        ([1],           1, 1),
        ([1],           2, 0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(subarray_sum)
print("subarray_sum defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Product Except Self — LC 238

---

```
PROBLEM:
  Return array where output[i] = product of all nums except nums[i].
  No division allowed. O(n) time, O(1) extra space.

TRICK:
  Left pass: output[i] = product of all nums to the left.
  Right pass: multiply each output[i] by running right product.
  No division, no extra array — result doubles as the left-product store.

SLOW MOTION TRACE on nums=[1,2,3,4]:
  Left pass (output[i] = product of nums[0..i-1]):
    output = [1, 1, 2, 6]   (1, 1*1, 1*2, 1*2*3)

  Right pass (right_running starts at 1, scan right to left):
    i=3: output[3] *= 1 → 6.   right_running = 1*4 = 4
    i=2: output[2] *= 4 → 8.   right_running = 4*3 = 12
    i=1: output[1] *= 12 → 12. right_running = 12*2 = 24
    i=0: output[0] *= 24 → 24. right_running = 24*1 = 24
  result = [24, 12, 8, 6]
  verify: [2*3*4, 1*3*4, 1*2*4, 1*2*3] = [24, 12, 8, 6] ✓

KEY INSIGHT:
  output[i] = (product of everything left) × (product of everything right).
  Two passes compute both without division.

TIME:  O(n) — two passes
SPACE: O(1) extra (output array doesn't count)
```

In [ ]:
def product_except_self(nums: List[int]) -> List[int]:
    """
    LC 238 — Product of Array Except Self
    Approach: left-product pass then right-product pass in-place.
    Args:
        nums (List[int]): integer array, guaranteed no division by zero issue.
    Returns:
        List[int]: output[i] = product of all nums except nums[i].
    Time:  O(n) — two passes
    Space: O(1) extra — output array is not counted as extra space
    """
    n = len(nums)
    output = [1] * n

    # Left pass: output[i] = product of nums[0..i-1]
    left_running = 1
    for i in range(n):
        output[i] = left_running           # everything to the left of i
        left_running *= nums[i]            # extend left product for next position

    # Right pass: multiply by product of nums[i+1..n-1]
    right_running = 1
    for i in range(n - 1, -1, -1):
        output[i] *= right_running         # multiply in the right side
        right_running *= nums[i]           # extend right product for next position

    return output

# Slow motion on [1,2,3,4]:
# left pass: output=[1,1,2,6]
# right pass: i=3→ *1=6, right=4; i=2→ *4=8, right=12; i=1→ *12=12, right=24; i=0→ *24=24
# result=[24,12,8,6]

def test_harness(fn):
    tests = [
        ([1,2,3,4],     [24,12,8,6]),
        ([-1,1,0,-3,3], [0,0,9,0,0]),
        ([1,0],         [0,1]),
        ([0,0],         [0,0]),
        ([2,3],         [3,2]),
        ([5],           [1]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(product_except_self)
print("product_except_self defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: 2D Prefix Sum — LC 304

---

```
PROBLEM:
  Answer multiple 2D range sum queries on a matrix.

TRICK:
  Build 2D prefix array pre where pre[r][c] = sum of all cells in
  the rectangle from (0,0) to (r-1, c-1).

  BUILD: pre[r][c] = grid[r-1][c-1] + pre[r-1][c] + pre[r][c-1] - pre[r-1][c-1]

  QUERY sum(r1,c1,r2,c2):
    = pre[r2+1][c2+1] - pre[r1][c2+1] - pre[r2+1][c1] + pre[r1][c1]

SLOW MOTION TRACE on 3x3 grid:
  grid = [[1,2,3],[4,5,6],[7,8,9]]
  pre (4x4, 0-padded border):
   0  0  0  0
   0  1  3  6
   0  5 12 21
   0 12 27 45

  sumRegion(1,1,2,2):
    = pre[3][3] - pre[1][3] - pre[3][1] + pre[1][1]
    = 45 - 6 - 12 + 1 = 28
  verify: 5+6+8+9 = 28 ✓

KEY INSIGHT:
  Inclusion-exclusion removes the double-subtracted corner.
  Same formula as 1D: subtract left, subtract top, add back corner.

TIME:  O(m*n) build, O(1) query
SPACE: O(m*n) — 2D prefix array
```

In [ ]:
class NumMatrix:
    """
    LC 304 — Range Sum Query 2D - Immutable
    Approach: 2D prefix sum with O(m*n) build, O(1) query.
    Time:  O(m*n) init, O(1) sumRegion
    Space: O(m*n) — 2D prefix array with 0-padded border
    """
    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.pre = [[0] * (n + 1) for _ in range(m + 1)]  # 0-padded border

        for r in range(1, m + 1):
            for c in range(1, n + 1):
                self.pre[r][c] = (
                    matrix[r-1][c-1]         # current cell
                    + self.pre[r-1][c]       # top rectangle
                    + self.pre[r][c-1]       # left rectangle
                    - self.pre[r-1][c-1]     # remove double-counted corner
                )

    def sumRegion(self, r1: int, c1: int, r2: int, c2: int) -> int:
        return (
            self.pre[r2+1][c2+1]     # full rectangle to bottom-right
            - self.pre[r1][c2+1]     # subtract top strip
            - self.pre[r2+1][c1]     # subtract left strip
            + self.pre[r1][c1]       # add back double-subtracted corner
        )

# Slow motion build on [[1,2,3],[4,5,6],[7,8,9]]:
# pre[1][1]=1, pre[1][2]=3, pre[1][3]=6
# pre[2][1]=5, pre[2][2]=12,pre[2][3]=21
# pre[3][1]=12,pre[3][2]=27,pre[3][3]=45
# sumRegion(1,1,2,2) = pre[3][3]-pre[1][3]-pre[3][1]+pre[1][1] = 45-6-12+1 = 28

def test_harness(cls):
    grid = [[3,0,1,4,2],[5,6,3,2,1],[1,2,0,1,5],[4,1,0,1,7],[1,0,3,0,5]]
    obj = cls(grid)
    tests = [
        (2,1,4,3, 8),
        (1,1,2,2, 11),
        (1,2,2,4, 12),
        (0,0,4,4, 58),
        (0,0,0,0, 3),
    ]
    passed = 0
    for r1,c1,r2,c2,expected in tests:
        got = obj.sumRegion(r1,c1,r2,c2)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | ({r1},{c1},{r2},{c2}) expected={expected} got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(NumMatrix)
print("NumMatrix defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Prefix XOR — Count Subarrays with Even XOR

---

```
PROBLEM:
  Count subarrays with XOR equal to 0 (or count even-XOR subarrays).

TRICK:
  Same idea as prefix sum: pre_xor[j] ^ pre_xor[i] = XOR(i..j-1)
  XOR(i..j) = 0  →  pre_xor[j+1] == pre_xor[i]
  Seed {0:1} for subarrays starting at index 0.

  XOR property: even XOR means pre_xor[j] has even parity.
  Count by parity: even[0]=1 (seed), even tracks even prefix XORs seen.

SLOW MOTION TRACE on nums=[1,2,3,4,5], count subarrays with XOR=0:
  pre_xor = [0, 1, 3, 0, 4, 1]
  Map of seen prefix XORs:
  j=0: pre=1, need=1^0=1, seen[1]=0, map={0:1,1:1}
  j=1: pre=3, need=3^0=3, seen[3]=0, map={0:1,1:1,3:1}
  j=2: pre=0, need=0^0=0, seen[0]=1, count+=1, map={0:2,...}
  j=3: pre=4, need=4^0=4, seen[4]=0, map={...4:1}
  j=4: pre=1, need=1^0=1, seen[1]=1, count+=1, map={...}
  answer=2  (subarrays [0,2] XOR=0 and [2,4] needs check)

KEY INSIGHT:
  XOR prefix works identically to sum prefix — just swap + with ^.
  The seed and complement formula are the same pattern.

TIME:  O(n) — one pass
SPACE: O(n) — prefix XOR HashMap
```

In [ ]:
from collections import defaultdict

def count_subarrays_xor_k(nums: List[int], k: int) -> int:
    """
    Count subarrays with XOR equal to k.
    Approach: prefix XOR + HashMap. XOR(i..j) = pre[j+1] ^ pre[i].
    Args:
        nums (List[int]): integer array.
        k (int): target XOR value.
    Returns:
        int: count of subarrays with XOR == k.
    Time:  O(n) — one pass
    Space: O(n) — prefix XOR HashMap
    """
    prefix_count = defaultdict(int)
    prefix_count[0] = 1   # seed: empty prefix XOR = 0, seen once
    running_xor = 0
    total = 0

    for v in nums:
        running_xor ^= v                    # extend prefix XOR
        complement = running_xor ^ k        # pre[i] = pre[j] ^ k means XOR(i..j)=k
        total += prefix_count[complement]   # count past prefixes matching complement
        prefix_count[running_xor] += 1

    return total

def count_even_xor_subarrays(nums: List[int]) -> int:
    """
    Count subarrays with even XOR (XOR == 0 or any even value).
    Approach: track parity of prefix XOR. Even XOR → same parity as running.
    Time:  O(n)
    Space: O(1) — just two counters
    """
    even = 1   # seed: prefix XOR=0 is even, seen once before any element
    odd = 0
    running_xor = 0
    total = 0

    for v in nums:
        running_xor ^= v
        if running_xor % 2 == 0:   # even prefix XOR
            total += even          # any past even prefix gives even subarray XOR
            even += 1
        else:
            total += odd
            odd += 1

    return total

# Slow motion on [1,2,3,4,5], k=0:
# pre_xor sequence: 0,1,3,0,4,1
# j=2: running=0, need=0^0=0, seen[0]=1 → count=1  (subarray [1,2,3])
# j=4: running=1, need=1^0=1, seen[1]=1 → count=2

def test_harness(fn):
    tests = [
        ([1,2,3,4,5], 0, 2),
        ([4,1,3],     0, 0),
        ([0,0,0],     0, 6),
        ([1],         1, 1),
        ([1,1],       0, 1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(count_subarrays_xor_k)
print("count_subarrays_xor_k and count_even_xor_subarrays defined.")

<a id='10'></a>

## 10. The Prefix Sum Decision Map

```
QUESTION TYPE                            KEY TECHNIQUE          LC PROBLEMS
──────────────────────────────────────────────────────────────────────────────
Range sum query (static array)           Prefix array            303, 307
Count subarrays summing to k             Prefix + HashMap        560, 974
Product except self (no division)        Left × right pass       238
2D range sum query                       2D prefix array         304
Count subarrays with XOR = k             Prefix XOR + HashMap    1310 style
Subarray sum divisible by k              Prefix mod k + HashMap  974

THE UNIVERSAL FORMULA:
  subarray property (i..j) = f(prefix[j+1]) - f(prefix[i])
  where f is sum, XOR, product, or any aggregation.
  HashMap: need[pre[j+1]] to find matching past pre[i].
  Seed: map[identity_element] = 1  (0 for sum, 0 for XOR, 1 for product).
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to reach for Prefix Sum:**

| Signal | What To Do |
|--------|------------|
| Range sum / XOR query, static array | Build prefix array |
| Count subarrays with sum/XOR = k | Prefix + HashMap + seed |
| Product of all except self | Two-pass left×right |
| 2D rectangle sum | 2D prefix with border |

**2. Core operations — memorize these:**

```python
pre = [0] * (n + 1)                   # 0-indexed prefix, 1 extra for seed
for i, v in enumerate(nums): pre[i+1] = pre[i] + v
range_sum = pre[j+1] - pre[i]         # sum of nums[i..j]

# Prefix + HashMap seed
count_map = defaultdict(int)
count_map[0] = 1                      # ALWAYS seed with identity element
```

**3. Common templates:**

```python
# TEMPLATE 1: BUILD AND QUERY
pre = [0] + list(accumulate(nums))
def query(i, j): return pre[j+1] - pre[i]

# TEMPLATE 2: COUNT SUBARRAYS SUMMING TO K
from collections import defaultdict
count_map = defaultdict(int)
count_map[0] = 1   # seed
running = total = 0
for v in nums:
    running += v
    total += count_map[running - k]
    count_map[running] += 1

# TEMPLATE 3: 2D PREFIX
pre = [[0]*(n+1) for _ in range(m+1)]
for r in range(1,m+1):
    for c in range(1,n+1):
        pre[r][c] = grid[r-1][c-1] + pre[r-1][c] + pre[r][c-1] - pre[r-1][c-1]
def query2d(r1,c1,r2,c2):
    return pre[r2+1][c2+1] - pre[r1][c2+1] - pre[r2+1][c1] + pre[r1][c1]
```

**4. Gotchas:**

```
❌  Forget seed {0:1} — misses subarrays starting at index 0
❌  Query as pre[j]-pre[i] instead of pre[j+1]-pre[i] — off by one
❌  2D query without the + corner — double-subtraction bug
❌  Build prefix without the leading 0 — i=0 boundary broken
✅  Prefix works with any aggregation: sum, XOR, product, max (with care)
✅  HashMap + prefix: swap + for ^ to switch sum → XOR mode
```

<a id='12'></a>

## 12. Summary Map

```
PREFIX SUM
│
├── 1D Array
│     ├── Build: pre[i+1] = pre[i] + nums[i]    O(n)
│     ├── Query: pre[j+1] - pre[i]               O(1)
│     └── LC 303 Range Sum Query
│
├── 1D + HashMap (count subarrays)
│     ├── pre[j] - pre[i] = k  →  need pre[j]-k in map
│     ├── Seed: map[0]=1 (critical)
│     └── LC 560 Subarray Sum = K
│
├── Two-Pass (no extra array)
│     ├── Left prefix × right suffix
│     └── LC 238 Product Except Self
│
├── 2D
│     ├── Build with inclusion-exclusion
│     └── LC 304 Range Sum Query 2D
│
└── Prefix XOR
      ├── Same structure, swap + → ^
      └── Even/odd parity counting

THE SEED IS ALWAYS THE IDENTITY ELEMENT:
  sum  → seed {0:1}
  XOR  → seed {0:1}
  product → seed would be {1:1}
```

---
*End of Prefix Sum Master Guide — Sean Edition*